# Chapter 18: Advanced Time Series Topics

Topics: infinite distributed lags, unit root testing, spurious regression, cointegration, error correction, forecasting.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.arima.model import ARIMA
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 18.3 Spurious Regression

## Step 2 – Spurious Regression

If $y_t$ and $x_t$ are **independent I(1) processes** (unrelated random walks), OLS on the levels regression $y_t = \beta_0 + \beta_1 x_t + u_t$ typically yields:

- A statistically significant $\hat{\beta}_1$ (spurious)
- High $R^2$ despite no true relationship
- Residuals that are non-stationary (strong serial correlation)

**Solution:** Always test for unit roots before regressing I(1) variables. If no cointegration exists, regress in **first differences** instead.

In [ ]:
np.random.seed(42)
T = 100
rw1 = np.cumsum(np.random.normal(0, 1, T))
rw2 = np.cumsum(np.random.normal(0, 1, T))

df_spurious = pd.DataFrame({'y': rw1, 'x': rw2})
m_spurious  = smf.ols('y ~ x', data=df_spurious).fit()

print(f"R² = {m_spurious.rsquared:.4f} (suspiciously high for unrelated random walks!)")
print(f"t-stat on x = {m_spurious.tvalues['x']:.4f}")
print(f"\nThis is spurious regression: both variables are I(1) random walks with no true relationship")

## 18.4 Cointegration Test

## Step 3 – Cointegration

Two I(1) series $y_t$ and $x_t$ are **cointegrated** if there exists $\beta$ such that:
$$z_t = y_t - \beta x_t \sim I(0)$$

This means they share a common stochastic trend and do not drift apart in the long run.

**Engle-Granger two-step test:**
1. Regress $y_t$ on $x_t$ (OLS in levels) → get residuals $\hat{z}_t$
2. ADF test on $\hat{z}_t$ (H₀: no cointegration)

If cointegrated, model using **Error Correction Model (ECM):**
$$\Delta y_t = \alpha_0 + \alpha_1 \Delta x_t + \gamma \hat{z}_{t-1} + e_t$$

where $\gamma < 0$ captures the speed of adjustment back to the long-run equilibrium.

In [ ]:
df = woo.dataWoo('inven')
y1 = df['inven'].values
y2 = df['gdp'].values

score, pval, crit = coint(y1, y2)
print(f"Engle-Granger cointegration test:")
print(f"  stat={score:.4f}, p-value={pval:.4f}")
print(f"  Critical values: {crit}")
print("→ Cointegrated" if pval < 0.05 else "→ Not cointegrated")

## 18.5 Forecasting with ARIMA

## Step 4 – ARIMA Forecasting

An **ARIMA(p,d,q)** model:
$$(1-\phi_1 L - \cdots - \phi_p L^p)(1-L)^d y_t = c + (1 + \theta_1 L + \cdots + \theta_q L^q)\varepsilon_t$$

where $L$ is the lag operator ($Ly_t = y_{t-1}$).

- $p$: AR order (autocorrelation structure)
- $d$: degree of differencing (to achieve stationarity)
- $q$: MA order (moving average)

**Model selection:** use ACF/PACF plots and information criteria (AIC, BIC).

In [ ]:
df = woo.dataWoo('phillips')
df.index = pd.date_range(start='1948', periods=len(df), freq='YE')

train = df['inf'][:-4]
test  = df['inf'][-4:]

arima = ARIMA(train, order=(1, 0, 1)).fit()
forecast = arima.forecast(steps=4)

plt.figure(figsize=(12, 5))
plt.plot(df['inf'], color='steelblue', label='Actual')
plt.plot(forecast.index, forecast, 'r--o', label='Forecast', lw=2)
plt.axvline(train.index[-1], color='gray', linestyle=':')
plt.title('ARIMA(1,0,1) Forecast: US Inflation')
plt.legend()
plt.savefig(f"{output_dir}/fig1_arima_forecast.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_18_Advanced_Time_Series_01.png'), dpi=150, bbox_inches='tight')
plt.show()

from sklearn.metrics import mean_squared_error
rmse = np.sqrt(mean_squared_error(test, forecast))
print(f"RMSE: {rmse:.4f}")

## VAR Model

## Step 5 – Vector Autoregression (VAR) and Impulse Response Functions

A **VAR(p)** models a system of $K$ time series jointly:
$$\mathbf{y}_t = \mathbf{c} + \mathbf{A}_1\mathbf{y}_{t-1} + \cdots + \mathbf{A}_p\mathbf{y}_{t-p} + \boldsymbol{\varepsilon}_t$$

where $\mathbf{y}_t$ is $(K\times 1)$, $\mathbf{A}_j$ are $(K\times K)$ matrices, $\boldsymbol{\varepsilon}_t \sim WN(\mathbf{0}, \boldsymbol{\Sigma})$.

**Impulse Response Function (IRF):** traces the dynamic effect of a one-unit shock in variable $j$ on variable $i$ over $h$ periods:
$$IRF_{ij}(h) = \frac{\partial y_{i,t+h}}{\partial \varepsilon_{j,t}} = [\boldsymbol{\Psi}_h]_{ij}$$

where $\boldsymbol{\Psi}_h = \sum_{j=1}^h \mathbf{A}_j\boldsymbol{\Psi}_{h-j}$.

In [ ]:
macro = df[['inf', 'unem']].dropna()
var_model = VAR(macro).fit(maxlags=4, ic='aic')
print(var_model.summary())

# Impulse Response Functions
irf = var_model.irf(10)
irf.plot(orth=False)
plt.savefig(f"{output_dir}/fig2_irf.png", dpi=150, bbox_inches='tight')
plt.savefig(os.path.join(output_dir, 'fig_Chapter_18_Advanced_Time_Series_02.png'), dpi=150, bbox_inches='tight')
plt.show()